In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [20]:
claims = pd.read_csv("claims.csv")
print(claims.info())
print (claims.shape)
claims.head()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 28 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Claim_ID                           15000 non-null  str    
 1   Claim_Form_Type                    15000 non-null  str    
 2   Provider_NPI                       15000 non-null  int64  
 3   Provider_Specialty                 15000 non-null  str    
 4   Payer                              15000 non-null  str    
 5   Place_of_Service_Code              15000 non-null  int64  
 6   Place_of_Service_Desc              15000 non-null  str    
 7   Diagnosis_Code                     15000 non-null  str    
 8   Procedure_Code                     15000 non-null  int64  
 9   Procedure_Desc                     15000 non-null  str    
 10  Modifier_Code                      6726 non-null   str    
 11  Billed_Amount                      15000 non-null  float64
 12  S

,Claim_ID,Claim_Form_Type,Provider_NPI,Provider_Specialty,Payer,Place_of_Service_Code,Place_of_Service_Desc,Diagnosis_Code,Procedure_Code,Procedure_Desc,...,Eligibility_Verified,Duplicate_Claim_Flag,NCCI_Bundling_Flag,Clearinghouse_Validation_Passed,Provider_Monthly_Claim_Volume,Provider_Payer_Prior_Denial_Count,Chronic_Condition_Flag,Claim_Status,CARC_Code,Denial_Reason
0,CLM100000,837P,3579337979,Orthopedics,Medicare,23,Emergency Room,I10,29881,Knee arthroscopy/surgery,...,True,False,False,True,80,3,False,Approved,NaN,NaN
1,CLM100001,837I,9409175431,Internal Medicine,Medicaid,21,Inpatient Hospital,J18.9,36415,Blood draw,...,True,False,False,True,257,1,False,Approved,NaN,NaN
2,CLM100002,837P,1396263773,Radiology,Medicaid,11,Office,E78.5,99213,"Office visit, low complexity",...,True,False,False,True,135,2,True,Approved,NaN,NaN
3,CLM100003,837I,8670024292,Orthopedics,Aetna,22,Outpatient Hospital,M54.5,93000,EKG,...,True,False,False,True,290,2,True,Denied,CO-45,Charge exceeds fee schedule/contracted amount
4,CLM100004,837I,7425344331,Orthopedics,Medicare,21,Inpatient Hospital,K21.9,99213,"Office visit, low complexity",...,True,False,False,True,187,2,False,Approved,NaN,NaN


In [21]:
claims.isnull().sum().sort_values(ascending=False)

CARC_Code                            12290
Denial_Reason                        12290
Modifier_Code                         8274
Claim_ID                                 0
Provider_Specialty                       0
Payer                                    0
Place_of_Service_Desc                    0
Place_of_Service_Code                    0
Diagnosis_Code                           0
Procedure_Code                           0
Provider_NPI                             0
Claim_Form_Type                          0
Billed_Amount                            0
Procedure_Desc                           0
Service_Date                             0
Claim_Submission_Date                    0
Prior_Auth_Required                      0
Prior_Auth_Obtained                      0
Days_to_Submission                       0
Payer_Timely_Filing_Limit_Days           0
Duplicate_Claim_Flag                     0
Eligibility_Verified                     0
NCCI_Bundling_Flag                       0
Clearinghou

## ## Handle Missing Values 
CARC_Code                            12290
Denial_Reason                        12290
Modifier_Code                         8274

In [22]:
claims["Modifier_Code"] = claims["Modifier_Code"].fillna("No_Modifier")
print(claims["Modifier_Code"].isnull().sum())
print(claims["Modifier_Code"].value_counts())
print(claims["Modifier_Code"].head(10))

0
Modifier_Code
No_Modifier    8274
59             2266
25             2206
76             1542
GT              712
Name: count, dtype: int64
0    No_Modifier
1             76
2    No_Modifier
3    No_Modifier
4    No_Modifier
5             59
6    No_Modifier
7    No_Modifier
8             59
9    No_Modifier
Name: Modifier_Code, dtype: str


## Instead of dropping Denial_Reason, keep it outside the training data.

In [23]:
# Keep a lookup table
carc_lookup = claims[["CARC_Code", "Denial_Reason"]].dropna().drop_duplicates()
print(carc_lookup.shape)
carc_lookup.head()

(8, 2)


,CARC_Code,Denial_Reason
3,CO-45,Charge exceeds fee schedule/contracted amount
5,CO-50,Not deemed a medical necessity
11,CO-97,Benefit bundled into another service (NCCI)
28,CO-18,Duplicate claim/service
37,CO-16,Missing/invalid info needed for adjudication


In [25]:
claims.drop( columns =["Claim_ID",
        "Provider_NPI"],
        inplace = True)
claims

,Claim_Form_Type,Provider_Specialty,Payer,Place_of_Service_Code,Place_of_Service_Desc,Diagnosis_Code,Procedure_Code,Procedure_Desc,Modifier_Code,Billed_Amount,...,Eligibility_Verified,Duplicate_Claim_Flag,NCCI_Bundling_Flag,Clearinghouse_Validation_Passed,Provider_Monthly_Claim_Volume,Provider_Payer_Prior_Denial_Count,Chronic_Condition_Flag,Claim_Status,CARC_Code,Denial_Reason
0,837P,Orthopedics,Medicare,23,Emergency Room,I10,29881,Knee arthroscopy/surgery,No_Modifier,3796.96,...,True,False,False,True,80,3,False,Approved,NaN,NaN
1,837I,Internal Medicine,Medicaid,21,Inpatient Hospital,J18.9,36415,Blood draw,76,27.95,...,True,False,False,True,257,1,False,Approved,NaN,NaN
2,837P,Radiology,Medicaid,11,Office,E78.5,99213,"Office visit, low complexity",No_Modifier,141.00,...,True,False,False,True,135,2,True,Approved,NaN,NaN
3,837I,Orthopedics,Aetna,22,Outpatient Hospital,M54.5,93000,EKG,No_Modifier,135.13,...,True,False,False,True,290,2,True,Denied,CO-45,Charge exceeds fee schedule/contracted amount
4,837I,Orthopedics,Medicare,21,Inpatient Hospital,K21.9,99213,"Office visit, low complexity",No_Modifier,180.85,...,True,False,False,True,187,2,False,Approved,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,837I,Neurology,Self-Pay,22,Outpatient Hospital,N39.0,72148,MRI lumbar spine,59,1905.88,...,True,False,False,True,151,0,True,Approved,NaN,NaN
14996,837P,Neurology,Self-Pay,23,Emergency Room,E78.5,71046,Chest X-ray,No_Modifier,104.37,...,True,False,False,True,111,0,False,Approved,NaN,NaN
14997,837I,Orthopedics,BCBS,21,Inpatient Hospital,I10,36415,Blood draw,GT,24.32,...,False,False,True,True,46,1,False,Denied,CO-45,Charge exceeds fee schedule/contracted amount
14998,837P,Dermatology,UnitedHealthcare,11,Office,I10,93000,EKG,No_Modifier,98.04,...,True,False,False,True,143,2,False,Approved,NaN,NaN


In [24]:
claims

,Claim_ID,Claim_Form_Type,Provider_NPI,Provider_Specialty,Payer,Place_of_Service_Code,Place_of_Service_Desc,Diagnosis_Code,Procedure_Code,Procedure_Desc,...,Eligibility_Verified,Duplicate_Claim_Flag,NCCI_Bundling_Flag,Clearinghouse_Validation_Passed,Provider_Monthly_Claim_Volume,Provider_Payer_Prior_Denial_Count,Chronic_Condition_Flag,Claim_Status,CARC_Code,Denial_Reason
0,CLM100000,837P,3579337979,Orthopedics,Medicare,23,Emergency Room,I10,29881,Knee arthroscopy/surgery,...,True,False,False,True,80,3,False,Approved,NaN,NaN
1,CLM100001,837I,9409175431,Internal Medicine,Medicaid,21,Inpatient Hospital,J18.9,36415,Blood draw,...,True,False,False,True,257,1,False,Approved,NaN,NaN
2,CLM100002,837P,1396263773,Radiology,Medicaid,11,Office,E78.5,99213,"Office visit, low complexity",...,True,False,False,True,135,2,True,Approved,NaN,NaN
3,CLM100003,837I,8670024292,Orthopedics,Aetna,22,Outpatient Hospital,M54.5,93000,EKG,...,True,False,False,True,290,2,True,Denied,CO-45,Charge exceeds fee schedule/contracted amount
4,CLM100004,837I,7425344331,Orthopedics,Medicare,21,Inpatient Hospital,K21.9,99213,"Office visit, low complexity",...,True,False,False,True,187,2,False,Approved,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,CLM114995,837I,4660502249,Neurology,Self-Pay,22,Outpatient Hospital,N39.0,72148,MRI lumbar spine,...,True,False,False,True,151,0,True,Approved,NaN,NaN
14996,CLM114996,837P,8894062374,Neurology,Self-Pay,23,Emergency Room,E78.5,71046,Chest X-ray,...,True,False,False,True,111,0,False,Approved,NaN,NaN
14997,CLM114997,837I,8013221213,Orthopedics,BCBS,21,Inpatient Hospital,I10,36415,Blood draw,...,False,False,True,True,46,1,False,Denied,CO-45,Charge exceeds fee schedule/contracted amount
14998,CLM114998,837P,9767528422,Dermatology,UnitedHealthcare,11,Office,I10,93000,EKG,...,True,False,False,True,143,2,False,Approved,NaN,NaN


In [26]:
claims.to_csv("claims_preprocessed_v1.csv", index=False)

In [27]:
import os

print(os.getcwd())  # Shows the folder where the file was saved

c:\Users\Saroon\Desktop\Phython\new_python\RCM


In [28]:
import os

os.listdir()

['.vscode',
 '01_Data_Understanding.ipynb',
 '02_Exploratory_Data_Analysis.ipynb',
 '03 – Data Preprocessing.ipynb',
 'claims.csv',
 'claims_preprocessed_v1.csv',
 'eda.csv',
 'rcm_claims_hard_synthetic.csv',
 'venv']